# 04 — Leakage-Safe Temporal Split and Scaling for Multi-Horizon Wind-Vector Forecasting

This notebook prepares the multi-scale wind-vector datasets generated by `03_feature_engineering.ipynb` for the modeling stage of:

**A Shamal-Transition-Aware Multi-Scale Residual Neural Network for Multi-Horizon Wind Vector Forecasting over the Persian Gulf and Gulf of Oman**

The chronological partition is retained from the original project:

- **Training:** 2005–2017
- **Validation:** 2018–2019
- **Independent test:** 2020

No random splitting is used.

Because the new forecasting problem predicts up to **24 h ahead**, the last 24 forecast-issue hours of 2017 and 2019 are excluded from their respective splits. This ensures that no training target enters the validation period and no validation target enters the 2020 test period.

The historical wind-vector predictors and future wind-vector targets are standardized with statistics learned **only from the training split**. Latitude and longitude are standardized separately. The cyclic calendar predictors (`hour_sin`, `hour_cos`, `doy_sin`, `doy_cos`) are already bounded and are therefore left unchanged.

The annual NetCDF feature files are not duplicated into a second large collection of scaled files. Instead, this notebook stores:

- the exact leakage-safe split manifest,
- fitted scalers,
- scaler statistics,
- model-data configuration,
- integrity checks.

Scaling will be applied batch-wise during model training and evaluation.


## 1. Imports and project paths

The notebook reads the vector feature datasets from:

`data/processed/vector_features/`

The temporal split manifest is written to:

`data/splits/vector/`

The fitted scalers are stored under:

`outputs/models/`

and reproducibility tables are stored under:

`outputs/tables/`

The lag and forecast-horizon definitions are read from the metadata produced by Notebook 03 rather than being redefined independently.


In [1]:
from pathlib import Path
import gc

import joblib
import numpy as np
import pandas as pd
import xarray as xr

from sklearn.preprocessing import StandardScaler


# ---------------------------------------------------------
# Locate the project root robustly
# ---------------------------------------------------------

CWD = Path.cwd()

if (CWD / "data").exists() and (CWD / "notebooks").exists():
    PROJECT_ROOT = CWD

elif CWD.name == "notebooks" and (CWD.parent / "data").exists():
    PROJECT_ROOT = CWD.parent

else:
    raise RuntimeError(
        "Could not locate the wind_paper project root. "
        "Run this notebook from the project root "
        "or from the notebooks directory."
    )


# ---------------------------------------------------------
# Project directories
# ---------------------------------------------------------

VECTOR_FEATURE_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vector_features"
)

VECTOR_SPLIT_DIR = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "vector"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "models"
)

TABLES_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
)


# ---------------------------------------------------------
# Metadata generated by Notebook 03
# ---------------------------------------------------------

TEMPORAL_SCALE_SCHEMA_PATH = (
    TABLES_DIR
    / "temporal_scale_schema.csv"
)

FORECAST_HORIZON_SCHEMA_PATH = (
    TABLES_DIR
    / "forecast_horizon_schema.csv"
)

VECTOR_MODEL_SCHEMA_PATH = (
    TABLES_DIR
    / "vector_model_schema.csv"
)

VECTOR_FEATURE_SUMMARY_PATH = (
    TABLES_DIR
    / "vector_feature_engineering_summary.csv"
)


# ---------------------------------------------------------
# Output files generated by this notebook
# ---------------------------------------------------------

SPLIT_MANIFEST_PATH = (
    VECTOR_SPLIT_DIR
    / "split_manifest.csv"
)

SPLIT_SUMMARY_PATH = (
    TABLES_DIR
    / "vector_data_split_summary.csv"
)

HISTORY_SCALER_PATH = (
    MODEL_DIR
    / "vector_history_scaler.joblib"
)

TARGET_SCALER_PATH = (
    MODEL_DIR
    / "vector_target_scaler.joblib"
)

SPATIAL_SCALER_PATH = (
    MODEL_DIR
    / "vector_spatial_scaler.joblib"
)

SCALER_STATS_PATH = (
    TABLES_DIR
    / "vector_scaler_statistics.csv"
)

MODEL_DATA_CONFIG_PATH = (
    TABLES_DIR
    / "vector_model_data_config.csv"
)

SCALING_CONFIG_PATH = (
    TABLES_DIR
    / "vector_scaling_config.csv"
)


# ---------------------------------------------------------
# Create output directories
# ---------------------------------------------------------

VECTOR_SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("Project root:")
print(PROJECT_ROOT.resolve())

print("\nVector feature directory:")
print(VECTOR_FEATURE_DIR.resolve())

print("\nVector split directory:")
print(VECTOR_SPLIT_DIR.resolve())

print("\nModel directory:")
print(MODEL_DIR.resolve())

print("\nTables directory:")
print(TABLES_DIR.resolve())


Project root:
/home/mike/Desktop/wind_paper

Vector feature directory:
/home/mike/Desktop/wind_paper/data/processed/vector_features

Vector split directory:
/home/mike/Desktop/wind_paper/data/splits/vector

Model directory:
/home/mike/Desktop/wind_paper/outputs/models

Tables directory:
/home/mike/Desktop/wind_paper/outputs/tables


## 2. Load the feature schema and define the chronological split

Notebook 03 created the lag and forecast-horizon metadata used by the new vector forecasting problem.

This notebook reads those files and derives the maximum forecast horizon automatically.

The base split is:

\[
2005\text{–}2017 \rightarrow \text{training}
\]

\[
2018\text{–}2019 \rightarrow \text{validation}
\]

\[
2020 \rightarrow \text{independent test}.
\]

For a multi-horizon forecast, the split must be defined not only by the forecast issue time but also by the **latest target-valid time**.

For example, an issue time at `2017-12-31 12:00` would include a +24 h target in 2018. Such a sample must not be part of training.

Therefore, the final 24 issue hours of 2017 and 2019 are purged from the preceding split.


In [2]:
# ---------------------------------------------------------
# Load Notebook 03 metadata
# ---------------------------------------------------------

required_metadata_files = [
    TEMPORAL_SCALE_SCHEMA_PATH,
    FORECAST_HORIZON_SCHEMA_PATH,
    VECTOR_MODEL_SCHEMA_PATH,
    VECTOR_FEATURE_SUMMARY_PATH,
]

for path in required_metadata_files:
    assert path.exists(), (
        f"Missing Notebook 03 metadata file: {path}"
    )


temporal_scale_schema = pd.read_csv(
    TEMPORAL_SCALE_SCHEMA_PATH
)

forecast_horizon_schema = pd.read_csv(
    FORECAST_HORIZON_SCHEMA_PATH
)

vector_model_schema = pd.read_csv(
    VECTOR_MODEL_SCHEMA_PATH
)

vector_feature_summary = pd.read_csv(
    VECTOR_FEATURE_SUMMARY_PATH
)


# ---------------------------------------------------------
# Recover exact lag and horizon definitions
# ---------------------------------------------------------

ALL_LAGS = sorted(
    temporal_scale_schema["lag_hours"]
    .astype(int)
    .unique()
    .tolist()
)

FORECAST_HORIZONS = sorted(
    forecast_horizon_schema["horizon_hours"]
    .astype(int)
    .unique()
    .tolist()
)

MAX_HISTORY_HOURS = max(
    ALL_LAGS
)

MAX_FORECAST_HOURS = max(
    FORECAST_HORIZONS
)


# ---------------------------------------------------------
# Chronological split definition
# ---------------------------------------------------------

TRAIN_YEARS = list(
    range(2005, 2018)
)

VALIDATION_YEARS = [
    2018,
    2019,
]

TEST_YEARS = [
    2020,
]


TRAIN_PERIOD_END = pd.Timestamp(
    "2017-12-31 23:00:00"
)

VALIDATION_PERIOD_END = pd.Timestamp(
    "2019-12-31 23:00:00"
)

TEST_PERIOD_END = pd.Timestamp(
    "2020-12-31 23:00:00"
)


# ---------------------------------------------------------
# Scaler fitting chunk size
# ---------------------------------------------------------

SCALER_CHUNK_HOURS = 168


print("Historical lags:")
print(ALL_LAGS)

print("\nForecast horizons:")
print(FORECAST_HORIZONS)

print(
    "\nMaximum historical context:",
    f"{MAX_HISTORY_HOURS} h"
)

print(
    "Maximum forecast horizon:",
    f"{MAX_FORECAST_HOURS} h"
)

print(
    "\nTraining years:",
    f"{TRAIN_YEARS[0]}–{TRAIN_YEARS[-1]}"
)

print(
    "Validation years:",
    f"{VALIDATION_YEARS[0]}–{VALIDATION_YEARS[-1]}"
)

print(
    "Test year:",
    TEST_YEARS[0]
)

print(
    "\nScaler chunk size:",
    f"{SCALER_CHUNK_HOURS} h"
)


Historical lags:
[0, 1, 2, 3, 6, 12, 24, 48, 72, 120, 168]

Forecast horizons:
[1, 3, 6, 12, 24]

Maximum historical context: 168 h
Maximum forecast horizon: 24 h

Training years: 2005–2017
Validation years: 2018–2019
Test year: 2020

Scaler chunk size: 168 h


## 3. Validate the vector feature datasets

Before defining the split, the annual model-ready feature files are checked against the Notebook 03 schema.

The validation confirms that:

- all annual files from 2005 through 2020 exist,
- every file contains 512 study cells,
- the lag dimension exactly matches the multi-scale lag schema,
- the horizon dimension exactly matches 1, 3, 6, 12, and 24 h,
- all historical-vector and target-vector variables are present,
- calendar predictors are present,
- issue times are chronological and unique.

This notebook assumes Notebook 03 has already completed successfully.


In [3]:
# ---------------------------------------------------------
# Discover vector-feature files
# ---------------------------------------------------------

expected_years = list(
    range(2005, 2021)
)

vector_feature_files = sorted(
    VECTOR_FEATURE_DIR.glob("*.nc")
)

found_years = [
    int(file.stem)
    for file in vector_feature_files
]


assert found_years == expected_years, (
    "Vector feature files do not match the expected "
    "2005–2020 sequence. Run Notebook 03 first."
)


REQUIRED_VARIABLES = [
    "u10_history",
    "v10_history",
    "u10_target",
    "v10_target",
    "hour_sin",
    "hour_cos",
    "doy_sin",
    "doy_cos",
]


for file in vector_feature_files:

    year = int(file.stem)

    with xr.open_dataset(file) as ds:

        assert ds.sizes["study_id"] == 512

        assert np.array_equal(
            ds["lag"].values.astype(int),
            np.asarray(
                ALL_LAGS,
                dtype=int
            )
        )

        assert np.array_equal(
            ds["horizon"].values.astype(int),
            np.asarray(
                FORECAST_HORIZONS,
                dtype=int
            )
        )

        for variable in REQUIRED_VARIABLES:
            assert variable in ds.data_vars, (
                f"{variable} missing from {file.name}"
            )

        times = pd.DatetimeIndex(
            pd.to_datetime(
                ds["valid_time"].values
            )
        )

        assert times.is_monotonic_increasing
        assert not times.has_duplicates


print(
    "Vector feature dataset validation: PASSED"
)

print(
    "Annual vector-feature files:",
    len(vector_feature_files)
)


Vector feature dataset validation: PASSED
Annual vector-feature files: 16


## 4. Build the leakage-safe split manifest

Each annual NetCDF file remains in `data/processed/vector_features/`.

The split manifest records the exact forecast issue-time window from each file that belongs to training, validation, or testing.

The final issue time of a split is selected so that:

\[
t + h_{\max}
\]

remains inside the same chronological partition.

Historical predictors are allowed to extend backward into an earlier period because they are information that would already be available at forecast issue time. Future targets are never allowed to cross forward into the next split.

This is the key distinction required for leakage-free multi-horizon evaluation.


In [4]:
# ---------------------------------------------------------
# Helper: assign split by year
# ---------------------------------------------------------

def split_name_for_year(year):

    if year in TRAIN_YEARS:
        return "train"

    if year in VALIDATION_YEARS:
        return "validation"

    if year in TEST_YEARS:
        return "test"

    raise ValueError(
        f"Unexpected year: {year}"
    )


# ---------------------------------------------------------
# Build manifest
# ---------------------------------------------------------

manifest_rows = []


for year in expected_years:

    file = (
        VECTOR_FEATURE_DIR
        / f"{year}.nc"
    )

    split = split_name_for_year(
        year
    )

    with xr.open_dataset(file) as ds:

        available_times = pd.DatetimeIndex(
            pd.to_datetime(
                ds["valid_time"].values
            )
        )

        available_start = available_times[0]
        available_end = available_times[-1]

        issue_start = available_start
        issue_end = available_end

        if year == TRAIN_YEARS[-1]:

            latest_allowed_issue = (
                TRAIN_PERIOD_END
                - pd.Timedelta(
                    hours=MAX_FORECAST_HOURS
                )
            )

            issue_end = min(
                issue_end,
                latest_allowed_issue
            )

        if year == VALIDATION_YEARS[-1]:

            latest_allowed_issue = (
                VALIDATION_PERIOD_END
                - pd.Timedelta(
                    hours=MAX_FORECAST_HOURS
                )
            )

            issue_end = min(
                issue_end,
                latest_allowed_issue
            )

        if year == TEST_YEARS[-1]:

            latest_allowed_issue = (
                TEST_PERIOD_END
                - pd.Timedelta(
                    hours=MAX_FORECAST_HOURS
                )
            )

            issue_end = min(
                issue_end,
                latest_allowed_issue
            )

        selected_times = available_times[
            (available_times >= issue_start)
            & (available_times <= issue_end)
        ]

        assert len(selected_times) > 0

        split_period_end = (
            TRAIN_PERIOD_END
            if split == "train"
            else VALIDATION_PERIOD_END
            if split == "validation"
            else TEST_PERIOD_END
        )

        assert (
            selected_times[-1]
            + pd.Timedelta(
                hours=MAX_FORECAST_HOURS
            )
            <= split_period_end
        )

        number_of_hours = len(
            selected_times
        )

        number_of_cells = (
            ds.sizes["study_id"]
        )

        manifest_rows.append({
            "year": year,
            "split": split,
            "feature_file": str(
                file.relative_to(
                    PROJECT_ROOT
                )
            ),
            "available_start": available_start,
            "available_end": available_end,
            "issue_start": selected_times[0],
            "issue_end": selected_times[-1],
            "earliest_target_time": (
                selected_times[0]
                + pd.Timedelta(
                    hours=min(
                        FORECAST_HORIZONS
                    )
                )
            ),
            "latest_target_time": (
                selected_times[-1]
                + pd.Timedelta(
                    hours=MAX_FORECAST_HOURS
                )
            ),
            "hours": number_of_hours,
            "study_cells": number_of_cells,
            "samples": (
                number_of_hours
                * number_of_cells
            ),
            "lags": len(
                ALL_LAGS
            ),
            "horizons": len(
                FORECAST_HORIZONS
            ),
        })


split_manifest = pd.DataFrame(
    manifest_rows
)


split_manifest.to_csv(
    SPLIT_MANIFEST_PATH,
    index=False
)

split_manifest.to_csv(
    SPLIT_SUMMARY_PATH,
    index=False
)


print(
    "Leakage-safe split manifest:"
)

display(
    split_manifest
)


print(
    "\nSaved:"
)

print(
    SPLIT_MANIFEST_PATH.resolve()
)

print(
    SPLIT_SUMMARY_PATH.resolve()
)


Leakage-safe split manifest:


,year,split,feature_file,available_start,available_end,issue_start,issue_end,earliest_target_time,latest_target_time,hours,study_cells,samples,lags,horizons
0,2005,train,data/processed/vector_features/2005.nc,2005-01-08,2005-12-31 23:00:00,2005-01-08,2005-12-31 23:00:00,2005-01-08 01:00:00,2006-01-01 23:00:00,8592,512,4399104,11,5
1,2006,train,data/processed/vector_features/2006.nc,2006-01-01,2006-12-31 23:00:00,2006-01-01,2006-12-31 23:00:00,2006-01-01 01:00:00,2007-01-01 23:00:00,8760,512,4485120,11,5
2,2007,train,data/processed/vector_features/2007.nc,2007-01-01,2007-12-31 23:00:00,2007-01-01,2007-12-31 23:00:00,2007-01-01 01:00:00,2008-01-01 23:00:00,8760,512,4485120,11,5
3,2008,train,data/processed/vector_features/2008.nc,2008-01-01,2008-12-31 23:00:00,2008-01-01,2008-12-31 23:00:00,2008-01-01 01:00:00,2009-01-01 23:00:00,8784,512,4497408,11,5
4,2009,train,data/processed/vector_features/2009.nc,2009-01-01,2009-12-31 23:00:00,2009-01-01,2009-12-31 23:00:00,2009-01-01 01:00:00,2010-01-01 23:00:00,8760,512,4485120,11,5
5,2010,train,data/processed/vector_features/2010.nc,2010-01-01,2010-12-31 23:00:00,2010-01-01,2010-12-31 23:00:00,2010-01-01 01:00:00,2011-01-01 23:00:00,8760,512,4485120,11,5
6,2011,train,data/processed/vector_features/2011.nc,2011-01-01,2011-12-31 23:00:00,2011-01-01,2011-12-31 23:00:00,2011-01-01 01:00:00,2012-01-01 23:00:00,8760,512,4485120,11,5
7,2012,train,data/processed/vector_features/2012.nc,2012-01-01,2012-12-31 23:00:00,2012-01-01,2012-12-31 23:00:00,2012-01-01 01:00:00,2013-01-01 23:00:00,8784,512,4497408,11,5
8,2013,train,data/processed/vector_features/2013.nc,2013-01-01,2013-12-31 23:00:00,2013-01-01,2013-12-31 23:00:00,2013-01-01 01:00:00,2014-01-01 23:00:00,8760,512,4485120,11,5
9,2014,train,data/processed/vector_features/2014.nc,2014-01-01,2014-12-31 23:00:00,2014-01-01,2014-12-31 23:00:00,2014-01-01 01:00:00,2015-01-01 23:00:00,8760,512,4485120,11,5



Saved:
/home/mike/Desktop/wind_paper/data/splits/vector/split_manifest.csv
/home/mike/Desktop/wind_paper/outputs/tables/vector_data_split_summary.csv


## 5. Verify chronological separation between training, validation, and test

The split is now checked using the **latest target-valid time**, not merely the issue year.

The required ordering is:

\[
\max(T_{\text{train target}})
<
\min(T_{\text{validation issue}})
\]

and:

\[
\max(T_{\text{validation target}})
<
\min(T_{\text{test issue}}).
\]

This guarantees that a +24 h target from an earlier split cannot appear as a future observation inside the next split.

The validation and test predictors may legitimately use historical observations from the immediately preceding period because those observations are causally available at forecast issue time.


In [5]:
for column in [
    "available_start",
    "available_end",
    "issue_start",
    "issue_end",
    "earliest_target_time",
    "latest_target_time",
]:
    split_manifest[column] = pd.to_datetime(
        split_manifest[column]
    )


train_rows = (
    split_manifest[
        split_manifest["split"] == "train"
    ]
)

validation_rows = (
    split_manifest[
        split_manifest["split"] == "validation"
    ]
)

test_rows = (
    split_manifest[
        split_manifest["split"] == "test"
    ]
)


train_latest_target = (
    train_rows["latest_target_time"]
    .max()
)

validation_first_issue = (
    validation_rows["issue_start"]
    .min()
)

validation_latest_target = (
    validation_rows["latest_target_time"]
    .max()
)

test_first_issue = (
    test_rows["issue_start"]
    .min()
)


assert (
    train_latest_target
    < validation_first_issue
), (
    "Training targets overlap the validation period."
)

assert (
    validation_latest_target
    < test_first_issue
), (
    "Validation targets overlap the test period."
)


print("Chronological target separation: PASSED")

print(
    "\nLatest training target:",
    train_latest_target
)

print(
    "First validation issue:",
    validation_first_issue
)

print(
    "\nLatest validation target:",
    validation_latest_target
)

print(
    "First test issue:",
    test_first_issue
)


split_totals = (
    split_manifest
    .groupby(
        "split",
        as_index=False
    )
    .agg(
        years=("year", "count"),
        hours=("hours", "sum"),
        samples=("samples", "sum"),
    )
)


print(
    "\nSplit totals:"
)

display(
    split_totals
)


Chronological target separation: PASSED

Latest training target: 2017-12-31 23:00:00
First validation issue: 2018-01-01 00:00:00

Latest validation target: 2019-12-31 23:00:00
First test issue: 2020-01-01 00:00:00

Split totals:


,split,years,hours,samples
0,test,1,8760,4485120
1,train,13,113760,58245120
2,validation,2,17496,8957952


## 6. Fit leakage-free scalers using training data only

Three separate standardization objects are used.

### Wind-history scaler

A two-feature `StandardScaler` is fitted to:

\[
[u_{10},v_{10}]
\]

from all historical lags in the **training split only**.

The same component-wise transformation will later be applied to every historical lag.

### Wind-target scaler

A separate two-feature scaler is fitted to the future training targets:

\[
[u_{10,t+h},v_{10,t+h}]
\]

for all five forecast horizons.

Predictions will be inverse-transformed before physical evaluation in `m s⁻¹`.

### Spatial scaler

Latitude and longitude are standardized using the 512 study-grid locations.

The cyclic calendar variables are not standardized because they already lie approximately in `[-1, 1]`.

The scalers are fitted incrementally in temporal chunks so that the complete multi-year dataset does not need to be loaded into memory at once.


In [6]:
history_scaler = StandardScaler()

target_scaler = StandardScaler()

spatial_scaler = StandardScaler()


# ---------------------------------------------------------
# Fit spatial scaler on the unique study grid
# ---------------------------------------------------------

sample_feature_file = (
    VECTOR_FEATURE_DIR
    / "2005.nc"
)

with xr.open_dataset(
    sample_feature_file
) as ds_sample:

    spatial_matrix = np.column_stack([
        ds_sample["latitude"]
        .values
        .astype(np.float64),

        ds_sample["longitude"]
        .values
        .astype(np.float64),
    ])


assert spatial_matrix.shape == (
    512,
    2
)

spatial_scaler.fit(
    spatial_matrix
)


def iter_time_chunks(ds, issue_start, issue_end, chunk_hours):
    """
    Yield chronological issue-time chunks from one annual
    vector-feature dataset.
    """

    subset = ds.sel(
        valid_time=slice(
            issue_start,
            issue_end
        )
    )

    number_of_times = (
        subset.sizes["valid_time"]
    )

    for start_index in range(
        0,
        number_of_times,
        chunk_hours
    ):

        end_index = min(
            start_index + chunk_hours,
            number_of_times
        )

        yield subset.isel(
            valid_time=slice(
                start_index,
                end_index
            )
        )


history_scaler_rows_seen = 0
target_scaler_rows_seen = 0


for _, row in train_rows.iterrows():

    year = int(
        row["year"]
    )

    file = (
        PROJECT_ROOT
        / row["feature_file"]
    )

    issue_start = pd.Timestamp(
        row["issue_start"]
    )

    issue_end = pd.Timestamp(
        row["issue_end"]
    )

    print(
        f"Fitting scalers from {year} ..."
    )

    with xr.open_dataset(file) as ds:

        for chunk in iter_time_chunks(
            ds=ds,
            issue_start=issue_start,
            issue_end=issue_end,
            chunk_hours=SCALER_CHUNK_HOURS,
        ):

            u_history = (
                chunk["u10_history"]
                .values
                .astype(
                    np.float64,
                    copy=False
                )
                .reshape(-1)
            )

            v_history = (
                chunk["v10_history"]
                .values
                .astype(
                    np.float64,
                    copy=False
                )
                .reshape(-1)
            )

            assert (
                u_history.size
                == v_history.size
            )

            history_matrix = np.column_stack([
                u_history,
                v_history,
            ])

            assert np.isfinite(
                history_matrix
            ).all()

            history_scaler.partial_fit(
                history_matrix
            )

            history_scaler_rows_seen += (
                history_matrix.shape[0]
            )

            u_target = (
                chunk["u10_target"]
                .values
                .astype(
                    np.float64,
                    copy=False
                )
                .reshape(-1)
            )

            v_target = (
                chunk["v10_target"]
                .values
                .astype(
                    np.float64,
                    copy=False
                )
                .reshape(-1)
            )

            assert (
                u_target.size
                == v_target.size
            )

            target_matrix = np.column_stack([
                u_target,
                v_target,
            ])

            assert np.isfinite(
                target_matrix
            ).all()

            target_scaler.partial_fit(
                target_matrix
            )

            target_scaler_rows_seen += (
                target_matrix.shape[0]
            )

            del (
                u_history,
                v_history,
                history_matrix,
                u_target,
                v_target,
                target_matrix,
            )

        gc.collect()


print(
    "\nTraining-only scaler fitting: COMPLETED"
)

print(
    "History rows seen:",
    f"{history_scaler_rows_seen:,}"
)

print(
    "Target rows seen:",
    f"{target_scaler_rows_seen:,}"
)

print(
    "Spatial rows seen:",
    spatial_matrix.shape[0]
)


Fitting scalers from 2005 ...
Fitting scalers from 2006 ...
Fitting scalers from 2007 ...
Fitting scalers from 2008 ...
Fitting scalers from 2009 ...
Fitting scalers from 2010 ...
Fitting scalers from 2011 ...
Fitting scalers from 2012 ...
Fitting scalers from 2013 ...
Fitting scalers from 2014 ...
Fitting scalers from 2015 ...
Fitting scalers from 2016 ...
Fitting scalers from 2017 ...

Training-only scaler fitting: COMPLETED
History rows seen: 640,696,320
Target rows seen: 291,225,600
Spatial rows seen: 512


## 7. Save the fitted scalers and reproducibility metadata

The fitted scaling objects are saved so the exact same transformations can be reused during:

- baseline training,
- residual-network training,
- validation,
- independent 2020 testing,
- inverse transformation of model outputs.

The metadata tables record the means and scales learned from the training data.

No validation or test values are used to estimate these statistics.


In [7]:
joblib.dump(
    history_scaler,
    HISTORY_SCALER_PATH
)

joblib.dump(
    target_scaler,
    TARGET_SCALER_PATH
)

joblib.dump(
    spatial_scaler,
    SPATIAL_SCALER_PATH
)


scaler_statistics = pd.DataFrame([
    {
        "scaler": "history",
        "variable": "u10_history",
        "training_mean": history_scaler.mean_[0],
        "training_scale": history_scaler.scale_[0],
        "n_samples_seen": int(
            np.asarray(
                history_scaler.n_samples_seen_
            ).reshape(-1)[0]
        ),
    },
    {
        "scaler": "history",
        "variable": "v10_history",
        "training_mean": history_scaler.mean_[1],
        "training_scale": history_scaler.scale_[1],
        "n_samples_seen": int(
            np.asarray(
                history_scaler.n_samples_seen_
            ).reshape(-1)[0]
        ),
    },
    {
        "scaler": "target",
        "variable": "u10_target",
        "training_mean": target_scaler.mean_[0],
        "training_scale": target_scaler.scale_[0],
        "n_samples_seen": int(
            np.asarray(
                target_scaler.n_samples_seen_
            ).reshape(-1)[0]
        ),
    },
    {
        "scaler": "target",
        "variable": "v10_target",
        "training_mean": target_scaler.mean_[1],
        "training_scale": target_scaler.scale_[1],
        "n_samples_seen": int(
            np.asarray(
                target_scaler.n_samples_seen_
            ).reshape(-1)[0]
        ),
    },
    {
        "scaler": "spatial",
        "variable": "latitude",
        "training_mean": spatial_scaler.mean_[0],
        "training_scale": spatial_scaler.scale_[0],
        "n_samples_seen": int(
            np.asarray(
                spatial_scaler.n_samples_seen_
            ).reshape(-1)[0]
        ),
    },
    {
        "scaler": "spatial",
        "variable": "longitude",
        "training_mean": spatial_scaler.mean_[1],
        "training_scale": spatial_scaler.scale_[1],
        "n_samples_seen": int(
            np.asarray(
                spatial_scaler.n_samples_seen_
            ).reshape(-1)[0]
        ),
    },
])

scaler_statistics.to_csv(
    SCALER_STATS_PATH,
    index=False
)


model_data_config = pd.DataFrame([
    {
        "parameter": "forecast_problem",
        "value": "multi-horizon wind-vector forecasting",
    },
    {
        "parameter": "target_components",
        "value": "u10,v10",
    },
    {
        "parameter": "forecast_horizons_hours",
        "value": ",".join(
            map(
                str,
                FORECAST_HORIZONS
            )
        ),
    },
    {
        "parameter": "historical_lags_hours",
        "value": ",".join(
            map(
                str,
                ALL_LAGS
            )
        ),
    },
    {
        "parameter": "maximum_historical_lag_hours",
        "value": MAX_HISTORY_HOURS,
    },
    {
        "parameter": "maximum_forecast_horizon_hours",
        "value": MAX_FORECAST_HOURS,
    },
    {
        "parameter": "training_years",
        "value": "2005-2017",
    },
    {
        "parameter": "validation_years",
        "value": "2018-2019",
    },
    {
        "parameter": "test_year",
        "value": "2020",
    },
    {
        "parameter": "study_cells",
        "value": 512,
    },
    {
        "parameter": "boundary_purge_hours",
        "value": MAX_FORECAST_HOURS,
    },
    {
        "parameter": "random_split",
        "value": False,
    },
    {
        "parameter": "history_scaled",
        "value": True,
    },
    {
        "parameter": "target_scaled",
        "value": True,
    },
    {
        "parameter": "spatial_coordinates_scaled",
        "value": True,
    },
    {
        "parameter": "cyclic_calendar_features_scaled",
        "value": False,
    },
    {
        "parameter": "scaler_fit_period",
        "value": "training split only",
    },
])

model_data_config.to_csv(
    MODEL_DATA_CONFIG_PATH,
    index=False
)


scaling_config = pd.DataFrame([
    {
        "input_group": "wind_history",
        "variables": "u10_history,v10_history",
        "transformation": "StandardScaler",
        "fit_data": "training only",
        "saved_object": HISTORY_SCALER_PATH.name,
    },
    {
        "input_group": "future_vector_target",
        "variables": "u10_target,v10_target",
        "transformation": "StandardScaler",
        "fit_data": "training targets only",
        "saved_object": TARGET_SCALER_PATH.name,
    },
    {
        "input_group": "spatial",
        "variables": "latitude,longitude",
        "transformation": "StandardScaler",
        "fit_data": "512 study-grid coordinates",
        "saved_object": SPATIAL_SCALER_PATH.name,
    },
    {
        "input_group": "calendar",
        "variables": "hour_sin,hour_cos,doy_sin,doy_cos",
        "transformation": "none",
        "fit_data": "not applicable",
        "saved_object": "none",
    },
])

scaling_config.to_csv(
    SCALING_CONFIG_PATH,
    index=False
)


print("Saved scalers:")

print(
    "\n1.",
    HISTORY_SCALER_PATH.resolve()
)

print(
    "\n2.",
    TARGET_SCALER_PATH.resolve()
)

print(
    "\n3.",
    SPATIAL_SCALER_PATH.resolve()
)


print(
    "\nScaler statistics:"
)

display(
    scaler_statistics
)


print(
    "\nModel-data configuration:"
)

display(
    model_data_config
)


print(
    "\nScaling configuration:"
)

display(
    scaling_config
)


Saved scalers:

1. /home/mike/Desktop/wind_paper/outputs/models/vector_history_scaler.joblib

2. /home/mike/Desktop/wind_paper/outputs/models/vector_target_scaler.joblib

3. /home/mike/Desktop/wind_paper/outputs/models/vector_spatial_scaler.joblib

Scaler statistics:


,scaler,variable,training_mean,training_scale,n_samples_seen
0,history,u10_history,0.875059,3.479905,640696320
1,history,v10_history,-0.895949,3.512462,640696320
2,target,u10_target,0.874863,3.479686,291225600
3,target,v10_target,-0.896346,3.511734,291225600
4,spatial,latitude,26.113281,1.713589,512
5,spatial,longitude,54.131836,3.722484,512



Model-data configuration:


,parameter,value
0,forecast_problem,multi-horizon wind-vector forecasting
1,target_components,"u10,v10"
2,forecast_horizons_hours,"1,3,6,12,24"
3,historical_lags_hours,"0,1,2,3,6,12,24,48,72,120,168"
4,maximum_historical_lag_hours,168
5,maximum_forecast_horizon_hours,24
6,training_years,2005-2017
7,validation_years,2018-2019
8,test_year,2020
9,study_cells,512



Scaling configuration:


,input_group,variables,transformation,fit_data,saved_object
0,wind_history,"u10_history,v10_history",StandardScaler,training only,vector_history_scaler.joblib
1,future_vector_target,"u10_target,v10_target",StandardScaler,training targets only,vector_target_scaler.joblib
2,spatial,"latitude,longitude",StandardScaler,512 study-grid coordinates,vector_spatial_scaler.joblib
3,calendar,"hour_sin,hour_cos,doy_sin,doy_cos",none,not applicable,none


## 8. Define reusable batch transformation functions

The annual NetCDF files remain in physical units.

The following helper functions provide the exact transformations that the later model-training notebooks should apply to each batch.

The wind-history and target transformations preserve the original tensor shape.

Target predictions must be inverse-transformed before computing meteorological metrics such as component RMSE, vector RMSE, wind-speed error, and directional error.


In [8]:
def transform_vector_tensor(
    u_array,
    v_array,
    scaler,
):
    """
    Standardize paired u/v arrays while preserving shape.
    """

    u_array = np.asarray(
        u_array
    )

    v_array = np.asarray(
        v_array
    )

    assert (
        u_array.shape
        == v_array.shape
    )

    original_shape = (
        u_array.shape
    )

    paired = np.column_stack([
        u_array.reshape(-1),
        v_array.reshape(-1),
    ])

    transformed = scaler.transform(
        paired
    )

    u_scaled = (
        transformed[:, 0]
        .reshape(
            original_shape
        )
        .astype(
            np.float32
        )
    )

    v_scaled = (
        transformed[:, 1]
        .reshape(
            original_shape
        )
        .astype(
            np.float32
        )
    )

    return (
        u_scaled,
        v_scaled,
    )


def inverse_transform_vector_tensor(
    u_scaled,
    v_scaled,
    scaler,
):
    """
    Convert standardized paired u/v arrays back to physical units.
    """

    u_scaled = np.asarray(
        u_scaled
    )

    v_scaled = np.asarray(
        v_scaled
    )

    assert (
        u_scaled.shape
        == v_scaled.shape
    )

    original_shape = (
        u_scaled.shape
    )

    paired = np.column_stack([
        u_scaled.reshape(-1),
        v_scaled.reshape(-1),
    ])

    restored = scaler.inverse_transform(
        paired
    )

    u_physical = (
        restored[:, 0]
        .reshape(
            original_shape
        )
        .astype(
            np.float32
        )
    )

    v_physical = (
        restored[:, 1]
        .reshape(
            original_shape
        )
        .astype(
            np.float32
        )
    )

    return (
        u_physical,
        v_physical,
    )


def transform_spatial_coordinates(
    latitude,
    longitude,
    scaler,
):
    """
    Standardize latitude and longitude using the fitted spatial scaler.
    """

    latitude = np.asarray(
        latitude
    )

    longitude = np.asarray(
        longitude
    )

    assert (
        latitude.shape
        == longitude.shape
    )

    paired = np.column_stack([
        latitude.reshape(-1),
        longitude.reshape(-1),
    ])

    transformed = scaler.transform(
        paired
    )

    return (
        transformed[:, 0]
        .reshape(
            latitude.shape
        )
        .astype(
            np.float32
        ),

        transformed[:, 1]
        .reshape(
            longitude.shape
        )
        .astype(
            np.float32
        ),
    )


print(
    "Reusable transformation functions: READY"
)


Reusable transformation functions: READY


## 9. Transformation and inverse-transformation test

A small training batch is transformed with the saved scalers and then converted back to physical units.

The reconstructed arrays must match the original values within floating-point tolerance.

This verifies that the saved preprocessing objects can be safely used by subsequent training and evaluation notebooks.


In [9]:
history_scaler_check = joblib.load(
    HISTORY_SCALER_PATH
)

target_scaler_check = joblib.load(
    TARGET_SCALER_PATH
)

spatial_scaler_check = joblib.load(
    SPATIAL_SCALER_PATH
)


first_train_row = (
    train_rows
    .sort_values("year")
    .iloc[0]
)

test_file = (
    PROJECT_ROOT
    / first_train_row["feature_file"]
)

test_issue_start = pd.Timestamp(
    first_train_row["issue_start"]
)

test_issue_end = (
    test_issue_start
    + pd.Timedelta(
        hours=23
    )
)


with xr.open_dataset(
    test_file
) as ds_test:

    batch = (
        ds_test
        .sel(
            valid_time=slice(
                test_issue_start,
                test_issue_end
            )
        )
        .load()
    )


u_history_original = (
    batch["u10_history"]
    .values
)

v_history_original = (
    batch["v10_history"]
    .values
)

u_history_scaled, v_history_scaled = (
    transform_vector_tensor(
        u_history_original,
        v_history_original,
        history_scaler_check,
    )
)

u_history_restored, v_history_restored = (
    inverse_transform_vector_tensor(
        u_history_scaled,
        v_history_scaled,
        history_scaler_check,
    )
)


assert np.allclose(
    u_history_original,
    u_history_restored,
    rtol=1e-5,
    atol=1e-5,
)

assert np.allclose(
    v_history_original,
    v_history_restored,
    rtol=1e-5,
    atol=1e-5,
)


u_target_original = (
    batch["u10_target"]
    .values
)

v_target_original = (
    batch["v10_target"]
    .values
)

u_target_scaled, v_target_scaled = (
    transform_vector_tensor(
        u_target_original,
        v_target_original,
        target_scaler_check,
    )
)

u_target_restored, v_target_restored = (
    inverse_transform_vector_tensor(
        u_target_scaled,
        v_target_scaled,
        target_scaler_check,
    )
)


assert np.allclose(
    u_target_original,
    u_target_restored,
    rtol=1e-5,
    atol=1e-5,
)

assert np.allclose(
    v_target_original,
    v_target_restored,
    rtol=1e-5,
    atol=1e-5,
)


latitude_original = (
    batch["latitude"]
    .values
)

longitude_original = (
    batch["longitude"]
    .values
)

latitude_scaled, longitude_scaled = (
    transform_spatial_coordinates(
        latitude_original,
        longitude_original,
        spatial_scaler_check,
    )
)


assert np.isfinite(
    latitude_scaled
).all()

assert np.isfinite(
    longitude_scaled
).all()


print(
    "History transform/inverse-transform: PASSED"
)

print(
    "Target transform/inverse-transform: PASSED"
)

print(
    "Spatial transformation: PASSED"
)


History transform/inverse-transform: PASSED
Target transform/inverse-transform: PASSED
Spatial transformation: PASSED


## 10. Final leakage and reproducibility checks

The final checks verify that:

- only 2005–2017 data contributed to fitted wind scalers,
- the number of observations seen by each scaler matches the split manifest,
- training targets stop before validation begins,
- validation targets stop before the independent 2020 test begins,
- the test year remains completely excluded from model selection and scaler fitting,
- the saved scaler and metadata files exist,
- no random split has been introduced.

The scaler sample counts provide an additional audit trail:

\[
N_{\text{history scaler}}
=
N_{\text{training grid-time samples}}
\times
N_{\text{lags}}
\]

and:

\[
N_{\text{target scaler}}
=
N_{\text{training grid-time samples}}
\times
N_{\text{forecast horizons}}.
\]


In [10]:
print(
    "Running final Notebook 04 integrity checks ..."
)


assert (
    train_rows["year"]
    .astype(int)
    .tolist()
    == TRAIN_YEARS
)

assert (
    validation_rows["year"]
    .astype(int)
    .tolist()
    == VALIDATION_YEARS
)

assert (
    test_rows["year"]
    .astype(int)
    .tolist()
    == TEST_YEARS
)


assert (
    train_rows["latest_target_time"]
    .max()
    < validation_rows["issue_start"]
    .min()
)

assert (
    validation_rows["latest_target_time"]
    .max()
    < test_rows["issue_start"]
    .min()
)


training_grid_time_samples = int(
    train_rows["samples"]
    .sum()
)

expected_history_scaler_rows = (
    training_grid_time_samples
    * len(ALL_LAGS)
)

expected_target_scaler_rows = (
    training_grid_time_samples
    * len(FORECAST_HORIZONS)
)


actual_history_scaler_rows = int(
    np.asarray(
        history_scaler.n_samples_seen_
    )
    .reshape(-1)[0]
)

actual_target_scaler_rows = int(
    np.asarray(
        target_scaler.n_samples_seen_
    )
    .reshape(-1)[0]
)

actual_spatial_scaler_rows = int(
    np.asarray(
        spatial_scaler.n_samples_seen_
    )
    .reshape(-1)[0]
)


assert (
    actual_history_scaler_rows
    == expected_history_scaler_rows
), (
    "History scaler sample count does not match "
    "the training manifest."
)

assert (
    actual_target_scaler_rows
    == expected_target_scaler_rows
), (
    "Target scaler sample count does not match "
    "the training manifest."
)

assert (
    actual_spatial_scaler_rows
    == 512
)


for scaler in [
    history_scaler,
    target_scaler,
    spatial_scaler,
]:

    assert np.isfinite(
        scaler.mean_
    ).all()

    assert np.isfinite(
        scaler.scale_
    ).all()

    assert (
        scaler.scale_
        > 0
    ).all()


required_output_files = [
    SPLIT_MANIFEST_PATH,
    SPLIT_SUMMARY_PATH,
    HISTORY_SCALER_PATH,
    TARGET_SCALER_PATH,
    SPATIAL_SCALER_PATH,
    SCALER_STATS_PATH,
    MODEL_DATA_CONFIG_PATH,
    SCALING_CONFIG_PATH,
]

for path in required_output_files:

    assert path.exists(), (
        f"Missing required output: {path}"
    )


train_hours = int(
    train_rows["hours"]
    .sum()
)

validation_hours = int(
    validation_rows["hours"]
    .sum()
)

test_hours = int(
    test_rows["hours"]
    .sum()
)

train_samples = int(
    train_rows["samples"]
    .sum()
)

validation_samples = int(
    validation_rows["samples"]
    .sum()
)

test_samples = int(
    test_rows["samples"]
    .sum()
)


print(
    "Temporal split: PASSED"
)

print(
    "Cross-boundary target purge: PASSED"
)

print(
    "Training-only wind scaling: PASSED"
)

print(
    "Target scaling: PASSED"
)

print(
    "Spatial scaling: PASSED"
)

print(
    "Saved metadata: PASSED"
)


print(
    "\nFinal leakage-safe dataset configuration:"
)

print(
    "Training issue hours   :",
    f"{train_hours:,}"
)

print(
    "Validation issue hours :",
    f"{validation_hours:,}"
)

print(
    "Test issue hours       :",
    f"{test_hours:,}"
)

print(
    "\nTraining grid-time samples   :",
    f"{train_samples:,}"
)

print(
    "Validation grid-time samples :",
    f"{validation_samples:,}"
)

print(
    "Test grid-time samples       :",
    f"{test_samples:,}"
)

print(
    "\nHistorical lags:",
    len(ALL_LAGS),
    ALL_LAGS,
)

print(
    "Forecast horizons:",
    len(FORECAST_HORIZONS),
    FORECAST_HORIZONS,
)

print(
    "Vector targets:",
    "u10 and v10"
)

print(
    "Maximum forecast horizon:",
    f"{MAX_FORECAST_HOURS} h"
)

print(
    "Boundary purge:",
    f"{MAX_FORECAST_HOURS} h"
)

print(
    "Random split:",
    False
)

print(
    "\nFinal integrity check: PASSED"
)

print(
    "\n04_data_split_and_scaling.ipynb: COMPLETED"
)


Running final Notebook 04 integrity checks ...
Temporal split: PASSED
Cross-boundary target purge: PASSED
Training-only wind scaling: PASSED
Target scaling: PASSED
Spatial scaling: PASSED
Saved metadata: PASSED

Final leakage-safe dataset configuration:
Training issue hours   : 113,760
Validation issue hours : 17,496
Test issue hours       : 8,760

Training grid-time samples   : 58,245,120
Validation grid-time samples : 8,957,952
Test grid-time samples       : 4,485,120

Historical lags: 11 [0, 1, 2, 3, 6, 12, 24, 48, 72, 120, 168]
Forecast horizons: 5 [1, 3, 6, 12, 24]
Vector targets: u10 and v10
Maximum forecast horizon: 24 h
Boundary purge: 24 h
Random split: False

Final integrity check: PASSED

04_data_split_and_scaling.ipynb: COMPLETED


## Notebook 04 complete

The multi-horizon wind-vector dataset now has:

- a strictly chronological training/validation/test split,
- explicit protection against cross-boundary +24 h targets,
- training-only standardization for historical wind vectors,
- training-only standardization for future vector targets,
- standardized spatial coordinates,
- unchanged cyclic calendar predictors,
- reproducible saved scalers and split metadata.

The 2020 period remains the independent final test set.

The next scientific step should be the **Shamal regime and transition-label notebook**. Those labels should be generated from a literature-supported meteorological definition and then linked to this existing leakage-safe split before the final Shamal-aware model is trained.
